# DreamerV3 on Google Colab

This notebook trains [DreamerV3](https://github.com/danijar/dreamerv3) on a few example environments entirely inside Colab. It is tested against the free-tier GPU runtime.

**Runtime setup (important):**

1. Click `Runtime -> Change runtime type`.
2. Set `Hardware accelerator` to `GPU` (T4 is fine).
3. Run the cells below in order.

You can skip any environment section you don't want to try. The Minecraft section is optional and heavy — read its notes before running it.

## 1. Verify the runtime

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch the runtime type to GPU.'
import sys
print('Python:', sys.version)

## 2. Install DreamerV3 and shared dependencies

This takes ~2 minutes. JAX is installed with CUDA 12 wheels so that the agent trains on the Colab GPU.

In [ ]:
%pip install -q --upgrade pip
%pip install -q 'dreamerv3>=3.0.0' 'gymnasium>=0.29' tensorboard ruamel.yaml cloudpickle
%pip install -q --upgrade 'jax[cuda12]' 'jaxlib'

import jax
print('JAX devices:', jax.devices())

## 3. Clone the examples

Pull this repository so the example scripts are available on the Colab filesystem.

In [ ]:
import os
if not os.path.isdir('/content/rl-dreamer'):
    !git clone --depth 1 https://github.com/kuds/rl-dreamer.git /content/rl-dreamer
%cd /content/rl-dreamer
!ls examples

## 4. Launch TensorBoard (optional)

Run this cell to watch live training curves while the agent learns.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /root/logdir

## 5. CartPole (smoke test)

Trains in about a minute. Good check that the install actually works.

In [ ]:
!python examples/01_gym_cartpole.py \
    --logdir /root/logdir/cartpole \
    --run.steps 20000

## 6. DeepMind Control — Walker Walk (from pixels)

A classic continuous-control benchmark. Installs `dm_control` then trains for a short number of steps so the cell finishes on a free GPU. Increase `run.steps` for real results.

In [ ]:
%pip install -q dm_control
# Colab needs a software GL renderer for MuJoCo offscreen rendering.
import os
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
!python examples/02_dmc_walker.py \
    --logdir /root/logdir/dmc_walker \
    --run.steps 50000 \
    --batch_size 8

## 7. Atari — Pong

Installs the ALE ROMs via the Gymnasium `accept-rom-license` extra.

In [ ]:
%pip install -q 'gymnasium[atari,accept-rom-license]' ale-py

In [ ]:
!python examples/03_atari_pong.py \
    --logdir /root/logdir/atari_pong \
    --run.steps 100000 \
    --batch_size 8

## 8. Crafter

The 2D Minecraft-inspired benchmark — a good stand-in if you want Minecraft-like dynamics without the Java / MineRL setup.

In [ ]:
%pip install -q crafter
!python examples/04_crafter.py \
    --logdir /root/logdir/crafter \
    --run.steps 100000 \
    --batch_size 8

## 9. Minecraft via MineRL (optional, heavy)

DreamerV3's flagship Minecraft diamond task runs on top of [MineRL](https://minerl.readthedocs.io), which in turn launches a real Minecraft JVM.

**Caveats on Colab:**
- Requires Java 8 (installed below) and an `xvfb` display.
- The first episode takes several minutes as Minecraft boots.
- Training a diamond-collecting agent from scratch takes tens of millions of env steps — well beyond a single Colab session. Use this cell as a demonstration of the plumbing rather than expecting paper-quality results.

If the install is too slow or unstable for your Colab, skip straight to Crafter above.

In [ ]:
# Install Java 8 and xvfb for the MineRL backend.
!apt-get -qq update
!apt-get -qq install -y openjdk-8-jdk xvfb > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
!java -version

In [ ]:
%pip install -q git+https://github.com/minerllabs/minerl@v1.0.2

In [ ]:
# Launch the example under xvfb so MineRL has a display to render into.
!xvfb-run -a python examples/06_minecraft.py \
    --logdir /root/logdir/minecraft \
    --run.steps 20000 \
    --batch_size 4

## 10. Inspecting trained checkpoints

`scripts/evaluate.py` reloads a saved agent and runs deterministic rollouts.

In [ ]:
!python scripts/evaluate.py \
    --task gym_CartPole-v1 \
    --preset size1m \
    --logdir /root/logdir/cartpole \
    --episodes 5

## 11. Recording rollout videos

`scripts/record.py` writes one MP4 per episode to `<logdir>/videos/`. Install the `imageio` ffmpeg backend first, then run it with the same `--task` / `--preset` / `--logdir` you used for training.

In [ ]:
%pip install -q imageio imageio-ffmpeg
!python scripts/record.py \
    --task gym_CartPole-v1 \
    --preset size1m \
    --logdir /root/logdir/cartpole \
    --episodes 3 \
    --fps 30
!ls /root/logdir/cartpole/videos

In [ ]:
# Embed the first video inline so you can watch it without downloading.
import glob
from IPython.display import HTML
from base64 import b64encode

videos = sorted(glob.glob('/root/logdir/cartpole/videos/*.mp4'))
if not videos:
    print('No videos found - did the recording cell above run successfully?')
else:
    data = b64encode(open(videos[0], 'rb').read()).decode()
    HTML(f'<video width=480 controls src="data:video/mp4;base64,{data}"></video>')

## 12. Next steps

- Increase `--run.steps` and model preset (`--preset size25m` or `size50m`) for real results.
- Mount Google Drive and point `--logdir` at it so checkpoints survive runtime resets:

```python
from google.colab import drive
drive.mount('/content/drive')
# then pass --logdir /content/drive/MyDrive/dreamerv3/myrun
```

- Plug in your own environment using `examples/07_custom_env.py` as a template.
- Read [`docs/architecture.md`](../docs/architecture.md), [`docs/LEARNING.md`](../docs/LEARNING.md), and [`docs/tensorboard_guide.md`](../docs/tensorboard_guide.md) for background and metric explanations.